# Tutorial 16: Same exit wave, two detector propagators

Compare the default **Fraunhofer FFT** with optional **Rayleigh–Sommerfeld** propagation from exactly the same synthetic FTH sample exit wave. The object aperture contains a phase texture and a displaced reference aperture supplies the reference wave. No material database, external file, noise, or multislice rerun is needed.

Each distance produces full holograms, a **left-half FFT / right-half RS image**, a signed difference, and a ratio. Raw outputs are preserved; a second comparison explicitly matches orientation and total intensity to isolate changes in pattern shape.

Production notebooks and `HologramPipelineConfig` default to `detector_propagation_method="fraunhofer"`. This comparison deliberately evaluates both methods; RS remains opt-in everywhere else.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from time import perf_counter
import sys

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/scattering_calculator").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from IPython.display import Image, display
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, TwoSlopeNorm
from scipy.fft import fft2, fftshift, ifftshift
from scattering_calculator.simulation_pipelines.simulation_configuration import DetectorConfig

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
output_dir = repo_root / "outputs" / "detector_propagation_comparison"
output_dir.mkdir(parents=True, exist_ok=True)


## 1. Experimental geometry

Use **778.1 eV** at the Co L₃ edge ([LBNL X-Ray Data Booklet](https://xdb.lbl.gov/Section1/Periodic_Table/Co_Web_data.htm)), a **100 nm diameter object hole**, and a **375 × 375 detector with 80 × 80 µm pixels** (30 mm active width). The sensor pitch stays fixed at both distances.

The editable example distances are **10 cm and 20 cm**. The reference hole has an assumed **20 nm diameter**, centered **90 nm** from the object center along x. The source grid is independent of the detector grid and covers both apertures.

The FFT baseline uses linear small-angle detector mapping. RS uses actual source-to-pixel distances. Both use the same detector-pixel quadrature. Here `use_detector_pixel_footprint=False` uses pixel centers; enable the footprint option for finite-pixel integration and convergence checks. This camera is the effective detector after any hardware/software binning, not the previous 1300/4 specification.

In [ ]:
detector_propagation_method = "fraunhofer"  # Normal workflow default.
comparison_methods = (detector_propagation_method, "rayleigh_sommerfeld")
assert len(set(comparison_methods)) == 2

from scipy.constants import h, c, electron_volt
photon_energy_eV = 778.1
wavelength = h * c / (photon_energy_eV * electron_volt)
source_shape = (128, 128)
source_pitch = 2e-9                 # m; 256 nm source window
object_radius = 50e-9               # 100 nm diameter
reference_radius = 10e-9            # editable assumption: 20 nm diameter
reference_center = (90e-9, 0.0)      # editable assumption: (x, y), m
detector_shape = (375, 375)
detector_pitch = 80e-6              # m, fixed at both distances
distances = {"10 cm detector": 0.10, "20 cm detector": 0.20}
fft_padding_factor = 8
ratio_floor_fraction = 1e-4
use_detector_pixel_footprint = False

half_width = detector_shape[1] * detector_pitch / 2
max_angle = np.arctan2(half_width, min(distances.values()))
source_phase_step = source_pitch * np.sin(max_angle) / wavelength
assert source_phase_step < 0.5, "Source grid fails the per-axis kernel Nyquist bound."
assert abs(reference_center[0]) + reference_radius < source_shape[1]*source_pitch/2
assert object_radius < min(source_shape)*source_pitch/2
print(f"Co L3: {photon_energy_eV:g} eV, wavelength {wavelength*1e9:.4f} nm")
print(f"Detector: {detector_shape}, pitch {detector_pitch*1e6:g} µm, width {2*half_width*1e3:g} mm")
for label, distance in distances.items():
    angle = np.rad2deg(np.arctan2(half_width, distance))
    fringe_pitch = wavelength * distance / np.hypot(*reference_center)
    print(f"{label}: edge half-angle {angle:.2f}°, reference fringe pitch ≈ {fringe_pitch*1e6:.0f} µm ({fringe_pitch/detector_pitch:.1f} pixels)")


## 2. Construct the sample exit wave once

`exit_wave` is a coherent thin-sample transmission multiplied by plane illumination. It is passed unchanged to both detector models. Replace this cell with a saved complex exit wave and its physical pixel pitch to compare an experimental sample model.

In [ ]:
yy, xx = np.indices(source_shape, dtype=float)
x = (xx - source_shape[1] / 2) * source_pitch
y = (yy - source_shape[0] / 2) * source_pitch
object_aperture = (x*x + y*y <= object_radius**2)
reference_aperture = ((x-reference_center[0])**2 + (y-reference_center[1])**2 <= reference_radius**2)
texture = 0.55 * np.sin(2*np.pi*x / 35e-9) * np.cos(2*np.pi*y / 45e-9)
transmission = object_aperture * (0.8 + 0.15*np.cos(2*np.pi*y / 55e-9)) * np.exp(1j*texture)
transmission = transmission + reference_aperture * np.exp(0.35j)
exit_wave = np.asarray(transmission, dtype=complex)
exit_wave.setflags(write=False)
source_snapshot = exit_wave.copy()

source_extent = np.array([-source_shape[1]/2-.5, source_shape[1]/2-.5,
                          -source_shape[0]/2-.5, source_shape[0]/2-.5]) * source_pitch * 1e9
fig, axes = plt.subplots(1, 2, figsize=(8, 3.4), constrained_layout=True)
for ax, data, title, cmap in zip(axes, [abs(exit_wave), np.ma.masked_where(abs(exit_wave)==0, np.angle(exit_wave))],
                                ["One shared exit wave: amplitude", "Phase inside the apertures (rad)"], ["magma", "twilight"]):
    im = ax.imshow(data, origin="lower", extent=source_extent, cmap=cmap)
    ax.set(title=title, xlabel="x (nm)", ylabel="y (nm)")
    fig.colorbar(im, ax=ax)
fig.savefig(output_dir / "exit_wave.png", dpi=150)
display(Image(filename=str(output_dir / "exit_wave.png")))
plt.close(fig)


## 3. Run the actual detector configuration with both methods

The small adapter below provides `DetectorConfig` with the already computed exit wave, sample pitch and beam wavelength. It performs **no sample propagation**. The FFT's padded array is only its numerical far-field input; RS integrates the native source window.

Raw arrays retain each backend's existing normalization. The FFT path normalizes its reciprocal-grid intensity by the source intensity; RS uses source/detector pixel areas. Those raw totals are printed, not silently equated.

In [ ]:
beam = SimpleNamespace(wavelength=wavelength, wavevector=2*np.pi/wavelength, coherence_length=None)

def propagate_shared_exit_wave(method, distance):
    detector = DetectorConfig(
        detector_propagation_method=method,
        shape=detector_shape,
        pixel_size=detector_pitch,
        detector_center=tuple((np.asarray(detector_shape)-1)/2),
        sample_to_detector_distance=distance,
        ignore_flat_detector_curvature=(method == "fraunhofer"),
        use_detector_pixel_footprint=use_detector_pixel_footprint,
        detector_pixel_footprint_samples=3,
    )
    detector.setup()
    detector.calc_realspace_resolution(beam)
    padding = tuple((n*(fft_padding_factor-1)//2, n*(fft_padding_factor-1)//2) for n in source_shape)
    padded = np.pad(exit_wave, padding)
    wavefront = SimpleNamespace(
        exit_wave=exit_wave,
        exit_wave_for_farfield=padded,
        hologram=abs(fftshift(fft2(ifftshift(padded))))**2,
    )
    propagated_sample = SimpleNamespace(
        propagator_method="Scalar",
        SampleConfig=SimpleNamespace(real_space_pixel_size=source_pitch),
        IlluminationConfig=SimpleNamespace(beam_params=beam),
        return_wavefront=lambda: wavefront,
        return_scalar_wavefield=lambda: exit_wave,
    )
    start = perf_counter()
    detector.assign_propagated_wavefront(propagated_sample)
    detector.detect_hologram()
    elapsed = perf_counter() - start
    hologram = detector.return_ideal_hologram().copy()
    assert hologram.shape == detector_shape and np.isfinite(hologram).all() and hologram.min() >= 0
    np.testing.assert_array_equal(exit_wave, source_snapshot)
    return hologram, detector_pitch, elapsed

results = {}
for label, distance in distances.items():
    case = {"distance": distance}
    for method in comparison_methods:
        case[method], case["pitch"], seconds = propagate_shared_exit_wave(method, distance)
        print(f"{label:18s} | {method:20s} | {seconds:6.2f} s | raw sum {case[method].sum():.6g}")
    results[label] = case
print("Verified: both methods received the same unchanged complex exit wave.")


## 4. Raw split view, then shape difference and ratio

**Top row:** raw Fraunhofer, raw RS and their half-image composite, with one shared logarithmic color scale. No orientation or intensity corrections are applied here.

**Bottom row:** RS is reversed along both detector axes to account for its `exp(-ikr)` convention versus the legacy negative-exponent FFT. This reverses the display coordinates only; the source is never conjugated or changed. The centered odd-sized detector grid is symmetric so reversal corresponds exactly to `(X,Y) → (-X,-Y)`. The center column belongs to RS in the half-image composites.

Each aligned image is then divided by its own total intensity. The split view shares one scale. The signed difference is `(RS_normalized − FFT_normalized) / max(FFT_normalized)`. The ratio is `log2(RS_normalized / FFT_normalized)`; **zero means equal, +1 means twice as bright, −1 means half as bright**. Pixels below the stated floor in either image are masked gray. The ratio's displayed range is capped at ±2; stored ratios are not clipped.

This second row compares pattern shape, not absolute photon collection. It includes finite-distance effects, angular geometry, obliquity, and numerical quadrature/interpolation differences.

In [ ]:
metrics = {}
for label, case in results.items():
    fft_raw = case["fraunhofer"]
    rs_raw = case["rayleigh_sommerfeld"]
    fft_shape = fft_raw / fft_raw.sum()
    rs_aligned = rs_raw[::-1, ::-1]
    rs_shape = rs_aligned / rs_aligned.sum()
    split_column = detector_shape[1] // 2
    raw_split = np.concatenate([fft_raw[:, :split_column], rs_raw[:, split_column:]], axis=1)
    shape_split = np.concatenate([fft_shape[:, :split_column], rs_shape[:, split_column:]], axis=1)
    difference = (rs_shape - fft_shape) / fft_shape.max()
    floor = ratio_floor_fraction * max(fft_shape.max(), rs_shape.max())
    valid = (fft_shape > floor) & (rs_shape > floor)
    ratio = np.full(detector_shape, np.nan)
    np.divide(rs_shape, fft_shape, out=ratio, where=valid)
    log_ratio = np.ma.masked_invalid(np.log2(ratio))
    assert valid.any()
    relative_l2 = np.linalg.norm(rs_shape-fft_shape) / np.linalg.norm(fft_shape)
    metrics[label] = {"relative_l2": relative_l2, "raw_total_ratio": rs_raw.sum()/fft_raw.sum(),
                      "ratio_valid_fraction": valid.mean()}
    print(f"{label}: shape relative L2={relative_l2:.4f}, raw RS/FFT totals={metrics[label]['raw_total_ratio']:.4g}, ratio valid={valid.mean():.1%}")

    half_x = detector_shape[1] * case["pitch"] * .5e3
    half_y = detector_shape[0] * case["pitch"] * .5e3
    extent = [-half_x, half_x, -half_y, half_y]
    raw_peak = max(fft_raw.max(), rs_raw.max())
    raw_norm = LogNorm(vmin=raw_peak*1e-5, vmax=raw_peak)
    shape_peak = max(fft_shape.max(), rs_shape.max())
    shape_norm = LogNorm(vmin=shape_peak*1e-5, vmax=shape_peak)
    diff_limit = max(np.max(abs(difference)), 1e-12)
    ratio_cmap = plt.get_cmap("RdBu_r").copy()
    ratio_cmap.set_bad("0.75")
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    panels = [
        (fft_raw, "Fraunhofer: raw", "magma", raw_norm, "raw intensity"),
        (rs_raw, "Rayleigh–Sommerfeld: raw", "magma", raw_norm, "raw intensity"),
        (raw_split, "Raw halves: FFT | RS", "magma", raw_norm, "raw intensity"),
        (shape_split, "Aligned, unit-sum halves: FFT | RS", "magma", shape_norm, "fraction of total"),
        (difference, "Aligned difference: RS − FFT", "RdBu_r", TwoSlopeNorm(vmin=-diff_limit, vcenter=0, vmax=diff_limit), "difference / FFT peak"),
        (log_ratio, "Aligned ratio: log₂(RS / FFT)", ratio_cmap, TwoSlopeNorm(vmin=-2, vcenter=0, vmax=2), "log₂ ratio"),
    ]
    for ax, (data, title, cmap, norm, color_label) in zip(axes.flat, panels):
        image = ax.imshow(data, origin="lower", extent=extent, cmap=cmap, norm=norm)
        ax.set(title=title, xlabel="detector x (mm)", ylabel="detector y (mm)")
        fig.colorbar(image, ax=ax, shrink=.82, label=color_label)
    for ax in (axes[0, 2], axes[1, 0]):
        ax.axvline(0, color="cyan", lw=.8)
    fig.suptitle(f"{label.capitalize()} · z = {case['distance']*1e2:g} cm · same exit wave · shape relative L2 = {relative_l2:.3f}", fontsize=14)
    stem = label.replace(" ", "_")
    fig.savefig(output_dir / f"{stem}_comparison.png", dpi=150)
    display(Image(filename=str(output_dir / f"{stem}_comparison.png")))
    plt.close(fig)
    np.savez_compressed(output_dir / f"{stem}_comparison.npz", exit_wave=exit_wave,
                        source_pitch=source_pitch, wavelength=wavelength, distance=case["distance"],
                        detector_pitch=case["pitch"], fft_raw=fft_raw, rs_raw=rs_raw,
                        fft_shape=fft_shape, rs_shape_aligned=rs_shape, difference=difference,
                        ratio=ratio, ratio_valid=valid)


## 5. Central line profiles and interpretation

The 100 nm aperture is deep in the far-field regime at both 10 and 20 cm. Differences primarily reflect angular mapping/obliquity, the documented phase and normalization conventions, and numerical quadrature/interpolation. This comparison should not be interpreted as evidence of strong near-field effects for this experiment.

The farther detector has smaller angular coverage because its physical width stays fixed. Refine source pitch at fixed source extent, FFT padding, and detector footprint quadrature for quantitative convergence. Direct RS skips exactly zero source pixels in the opaque mask but retains all nonzero amplitudes without a threshold.

The selector controls only the **sample-to-detector** step. Multislice still uses exact-dispersion angular spectrum. A plain inverse FFT is not the RS detector adjoint; see [Light Propagation Modes](../docs/light_propagation_modes.md).

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(12, 3.4), constrained_layout=True)
for ax, (label, case) in zip(np.atleast_1d(axes), results.items()):
    fft_shape = case["fraunhofer"] / case["fraunhofer"].sum()
    rs_shape = case["rayleigh_sommerfeld"][::-1, ::-1] / case["rayleigh_sommerfeld"].sum()
    detector_x = (np.arange(detector_shape[1])-(detector_shape[1]-1)/2) * case["pitch"] * 1e3
    mid = detector_shape[0] // 2
    # Odd grids have a pixel centered at y=0; even grids straddle it.
    center_rows = slice(mid, mid+1) if detector_shape[0] % 2 else slice(mid-1, mid+1)
    ax.plot(detector_x, fft_shape[center_rows].mean(axis=0), label="Fraunhofer")
    ax.plot(detector_x, rs_shape[center_rows].mean(axis=0), label="RS, orientation aligned", alpha=.85)
    ax.set(title=label.capitalize(), xlabel="detector x (mm)", ylabel="unit-sum intensity", yscale="log")
    ax.grid(alpha=.2)
    ax.legend()
fig.savefig(output_dir / "central_profiles.png", dpi=150)
display(Image(filename=str(output_dir / "central_profiles.png")))
plt.close(fig)
print("Saved figures and raw/derived arrays to:", output_dir)


## 6. Radial averages

Average the detector pixels in concentric annuli centered on the optical axis. Each bin is one detector pixel wide; the mean is divided by the number of pixels in that annulus, so this is **mean intensity**, not annular integrated power. Only radii inside the largest inscribed circle are used, avoiding incomplete rings near the detector corners.

For each distance, the first panel compares raw radial averages and the second compares averages after normalizing each full hologram to unit total intensity. The last two panels show the signed difference and the **ratio of radial means**, not the radial mean of pixelwise ratios. The ratio is masked where the Fraunhofer radial mean is below `ratio_floor_fraction` of its radial peak.

The horizontal axis is physical scattering angle `atan(radius / distance)`, shared by both models. RS uses the same orientation alignment as above (a 180° reversal does not change radial averages). Radial averaging complements the two-dimensional plots: it can hide angular fringe differences.

In [ ]:
def radial_mean(image, pixel_pitch):
    """Pixel-center annular means on a centered detector, within complete rings."""
    ny, nx = image.shape
    yy, xx = np.indices(image.shape, dtype=float)
    radius_px = np.hypot(xx - (nx-1)/2, yy - (ny-1)/2)
    n_bins = min(ny, nx) // 2
    bin_index = np.floor(radius_px).astype(int)
    inside = bin_index < n_bins
    counts = np.bincount(bin_index[inside], minlength=n_bins)
    totals = np.bincount(bin_index[inside], weights=image[inside], minlength=n_bins)
    radii = np.bincount(bin_index[inside], weights=radius_px[inside], minlength=n_bins)
    keep = counts > 0
    return radii[keep] / counts[keep] * pixel_pitch, totals[keep] / counts[keep], counts[keep]

# Sanity checks: a constant image has a constant mean, and bins preserve
# integrated intensity within the circular region used for averaging.
test_image = np.ones(detector_shape)
_, test_means, test_counts = radial_mean(test_image, 1.0)
np.testing.assert_allclose(test_means, 1.0)

radial_results = {}
fig, axes = plt.subplots(len(results), 4, figsize=(17, 3.8*len(results)),
                         squeeze=False, constrained_layout=True)
for row, (label, case) in enumerate(results.items()):
    fft_raw = case["fraunhofer"]
    rs_raw = case["rayleigh_sommerfeld"][::-1, ::-1]
    radius, fft_mean, counts = radial_mean(fft_raw, case["pitch"])
    rs_radius, rs_mean, rs_counts = radial_mean(rs_raw, case["pitch"])
    np.testing.assert_array_equal(radius, rs_radius)
    np.testing.assert_array_equal(counts, rs_counts)
    np.testing.assert_allclose(radial_mean(rs_raw[::-1, ::-1], case["pitch"])[1], rs_mean)
    angle_deg = np.rad2deg(np.arctan2(radius, case["distance"]))
    fft_normalized = fft_mean / fft_raw.sum()
    rs_normalized = rs_mean / rs_raw.sum()
    radial_difference = rs_normalized - fft_normalized
    valid = fft_normalized > ratio_floor_fraction * fft_normalized.max()
    radial_ratio = np.full_like(fft_normalized, np.nan)
    np.divide(rs_normalized, fft_normalized, out=radial_ratio, where=valid)
    assert valid.any() and np.isfinite(radial_difference).all()

    for ax, fft_values, rs_values, title, ylabel in [
        (axes[row, 0], fft_mean, rs_mean, "Raw radial averages", "mean raw intensity"),
        (axes[row, 1], fft_normalized, rs_normalized, "Unit-sum radial averages", "mean unit-sum intensity"),
    ]:
        ax.semilogy(angle_deg, fft_values, label="Fraunhofer")
        ax.semilogy(angle_deg, rs_values, label="Rayleigh–Sommerfeld")
        ax.set(title=f"{label.capitalize()}\n{title}", ylabel=ylabel)
        ax.legend(fontsize=8)
    axes[row, 2].plot(angle_deg, radial_difference, color="tab:purple")
    axes[row, 2].axhline(0, color="0.4", lw=.8)
    axes[row, 2].set(title="Radial difference: RS − FFT", ylabel="difference of unit-sum means")
    axes[row, 3].plot(angle_deg, radial_ratio, color="tab:green")
    axes[row, 3].axhline(1, color="0.4", lw=.8, linestyle="--")
    axes[row, 3].set(title="Radial ratio: RS / FFT", ylabel="ratio of unit-sum means")
    for ax in axes[row]:
        ax.set_xlabel("scattering angle (degrees)")
        ax.grid(alpha=.2)
    radial_results[label] = dict(radius_m=radius, angle_deg=angle_deg, pixels_per_bin=counts,
                                fft_mean_raw=fft_mean, rs_mean_raw=rs_mean,
                                fft_mean_normalized=fft_normalized, rs_mean_normalized=rs_normalized,
                                difference=radial_difference, ratio=radial_ratio, ratio_valid=valid)
    np.savez_compressed(output_dir / f"{label.replace(' ', '_')}_radial_averages.npz", **radial_results[label])

fig.savefig(output_dir / "radial_averages.png", dpi=150)
display(Image(filename=str(output_dir / "radial_averages.png")))
plt.close(fig)
print("Saved radial averages, bin counts, differences and ratios to:", output_dir)
